In [ ]:
Celda 1: Descargar stocks.txt, configurar Spark y cargar el archivo (Paso 1)

In [4]:
import os
from pyspark.sql import SparkSession

# 1. Descargar el dataset stocks.txt dentro de la carpeta DATOS
!mkdir -p DATOS
!wget -q "https://github.com/josepgarcia/datos/raw/refs/heads/main/stocks.txt" -O ./DATOS/stocks.txt

# 2. Crear SparkSession personalizada ("AnalisisStocks" y 512m de memoria)
spark = (SparkSession.builder
    .appName("AnalisisStocks")
    .config("spark.executor.memory", "512m")
    .getOrCreate())

sc = spark.sparkContext

# 3. Construir ruta absoluta con prefijo file:// y cargar archivo
path_relativo = "./DATOS/stocks.txt"
path_absoluto = os.path.abspath(path_relativo)
path_final = "file://" + path_absoluto

rdd = sc.textFile(path_final)

# 4. Verificar carga mostrando las primeras 5 líneas
print("Primeras 5 líneas del archivo:")
for linea in rdd.take(5):
    print(linea)

Primeras 5 líneas del archivo:
id,name,category,quantity,price
1,iPhone,Electronics,10,899.99
2,Macbook,Electronics,5,1299.99
3,iPad,Electronics,15,499.99
4,Samsung TV,Electronics,8,799.99


In [ ]:
Limpieza y Parseo de Datos (Paso 2)

In [6]:
# Paso 2: Limpieza y parseo adaptado a la estructura real del archivo (id,name,category,quantity,price)
def parsear_linea(linea):
    try:
        partes = linea.split(",")
        simbolo = partes[2].strip()       # Categoría (ej. Electronics) como identificador de grupo
        fecha = f"ID-{partes[0].strip()}" # ID de registro
        precio = float(partes[4].strip()) # Precio (price)
        volumen = int(partes[3].strip())  # Cantidad (quantity)
        return (simbolo, fecha, precio, volumen)
    except:
        return None

# Aplicar función y filtrar líneas inválidas o encabezado
rdd_parseado = rdd.map(parsear_linea).filter(lambda x: x is not None)

print(f"Líneas válidas procesadas: {rdd_parseado.count()}")
print("Muestra de registros limpios:", rdd_parseado.take(3))

Líneas válidas procesadas: 30
Muestra de registros limpios: [('Electronics', 'ID-1', 899.99, 10), ('Electronics', 'ID-2', 1299.99, 5), ('Electronics', 'ID-3', 499.99, 15)]


In [ ]:
Paso 3 (Estadísticas por símbolo/categoría)

In [7]:
# Paso 3: Análisis por Símbolo/Categoría - Estadísticas Básicas
simbolo_precio = rdd_parseado.map(lambda x: (x[0], x[2]))

# 1. Precio máximo por símbolo
precio_max = simbolo_precio.reduceByKey(lambda a, b: max(a, b))

# 2. Precio mínimo por símbolo
precio_min = simbolo_precio.reduceByKey(lambda a, b: min(a, b))

# 3. Precio promedio usando aggregateByKey
precio_promedio = simbolo_precio.aggregateByKey(
    (0.0, 0),
    lambda acc, valor: (acc[0] + valor, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
).mapValues(lambda x: x[0] / x[1])

print("Precio máximo por categoría:")
for simbolo, precio in sorted(precio_max.collect()):
    print(f"  {simbolo}: ${precio:.2f}")

print("\nPrecio mínimo por categoría:")
for simbolo, precio in sorted(precio_min.collect()):
    print(f"  {simbolo}: ${precio:.2f}")

print("\nPrecio promedio por categoría:")
for simbolo, precio in sorted(precio_promedio.collect()):
    print(f"  {simbolo}: ${precio:.2f}")

Precio máximo por categoría:
  Accessories: $29.99
  Clothing: $199.99
  Electronics: $1299.99
  Food: $5.99
  Furniture: $249.99
  Sports: $49.99

Precio mínimo por categoría:
  Accessories: $24.99
  Clothing: $14.99
  Electronics: $129.99
  Food: $0.25
  Furniture: $79.99
  Sports: $19.99

Precio promedio por categoría:
  Accessories: $27.49
  Clothing: $99.28
  Electronics: $586.66
  Food: $2.30
  Furniture: $141.99
  Sports: $34.99


In [ ]:
Paso 4 (Top días/registros con mayor volumen):

In [8]:
# Paso 4: Análisis de Volumen - Top 10 de Mayor Actividad
fecha_volumen = rdd_parseado.map(lambda x: (x[1], x[3]))

# Sumar volumen por registro/fecha
volumen_por_fecha = fecha_volumen.reduceByKey(lambda a, b: a + b)

# Ordenar descendente y tomar top 10
top_fechas = volumen_por_fecha.sortBy(lambda x: x[1], ascending=False)

print("Top 10 registros con mayor volumen:")
for fecha, volumen in top_fechas.take(10):
    print(f"  {fecha}: {volumen:,} unidades")

Top 10 registros con mayor volumen:
  ID-13: 150 unidades
  ID-14: 120 unidades
  ID-12: 100 unidades
  ID-15: 50 unidades
  ID-29: 50 unidades
  ID-30: 40 unidades
  ID-28: 30 unidades
  ID-6: 30 unidades
  ID-16: 30 unidades
  ID-25: 30 unidades
